![alt text](STAR_MELT_logo.png "STAR_MELT")

# STAR-MELT Photosphere removal tool

In [ ]:
#Packages required by STAR-MELT, some are not used directly in this notebook, but called by the modules, check that they are all installed here
import time
from matplotlib import *
from matplotlib.pyplot import *
import numpy as np
import pandas as pd
import astropy
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SkyCoord, EarthLocation
from astropy.stats import sigma_clip
from astroquery.simbad import Simbad
from astropy.timeseries import LombScargle
from astropy.table import Table
import numpy.ma as ma
import os
from PyAstronomy import pyasl
from lmfit.models import GaussianModel, LinearModel
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit 
from scipy.signal import savgol_filter

from star_melt import *

from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widget
from IPython.display import display,clear_output,Image


#### If you change which %matplotlib magic command here you must restart the notebook

In [ ]:
#for notebook/slides
#%matplotlib notebook 
#for lab
%matplotlib widget
#for new window plots
#%matplotlib qt
%reload_ext autoreload
%autoreload 2

In [ ]:
rcParams.update({'figure.max_open_warning': 50})
rcParams['figure.dpi'] = 100
matplotlib.rc('font', family='sans',size=14)
USH.fig_size_s=(6,5)
USH.fig_size_l=(9,5)
USH.fig_size_n=(9,3)

line_table=USH.line_table
line_table=USH.line_table_prev_obs
spt_teff=USH.spt_teff

template_dir='/Users/jcampbel/Library/CloudStorage/OneDrive-ESO/standard_stars/'

data_dates_range_templ = USH.load_phot_templates(template_dir)


In [ ]:
data_dates_range_templ

### Select data_directory


In [ ]:

data_dir='/Users/jcampbel/Library/CloudStorage/OneDrive-ESO/PEN_data_final_copy_180923/'
#data_dir='change_me'

star_list=os.listdir(data_dir)
try:
    star_list.remove('.DS_Store')#remove temp file on mac because python will think it's a star!
except:
    pass

star_select = widget.Dropdown(
    options=sorted(star_list),
    #value='RY Tau',
    description='Select region:',
)
display(star_select)


### Read in data and see what data is available for directory

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
target=star_select.value
data_fits_files=get_files(os.path.join(data_dir,target),'.fits','.FTZ')

data_dates_range2,instrument,w0=get_instrument_date_details(data_fits_files,qgrid=True)
data_dates_range2.sort_values('wmin',inplace=True)


spts=[]
for target in data_dates_range2.target:
    #some common simbad query issues for getting sp_t estimate of target stars
    target=target.replace('O-','O')
    target=target.replace('EM','')
    target=target.replace('YL','Y L')
    if target.startswith('V '):
        target=target.replace('V ', '', 1)
    if target.startswith('SO'):
        target=target.replace('SO','HHM2007 ')
    try:
        simbad=customSimbad.query_object(target)
        mk_tar=simbad['sp_type'][0] #if this does not work try using 'SP_TYPE' in caps
        if mk_tar.startswith('d'):
            mk_tar = mk_tar[1:]#remove 'd' from start of sp_t
        if len(mk_tar) >= 3 and mk_tar[2] == '.':
            mk_tar = mk_tar[:4]
        else:
            mk_tar = mk_tar[:2]
        if target=='DI Cha':
            mk_tar='K0'
    except:
        mk_tar='K9'
        pass    
    spts.append(mk_tar)
data_dates_range2['sp_t']=spts

data_dates_range2

### Select target for interactive / individual removals

In [ ]:
target_select = widget.SelectMultiple(options=unique(data_dates_range2.target),value=[data_dates_range2.target[0]],description='Select target(s):',style = {'description_width': 'initial'})
display(target_select)

In [ ]:
data_dates_range1=data_dates_range2[data_dates_range2.target.isin(target_select.value)]
inst_options=['any']
inst_options.extend(unique(data_dates_range1.inst).tolist())
inst_select = widget.RadioButtons(
    options=inst_options,
    description='Instrument selection:',
    style = {'description_width': 'initial'}
)
display(inst_select)

In [ ]:
data_dates_range3=data_dates_range1
if inst_select.value!='any':
    data_dates_range3=data_dates_range1[data_dates_range1.inst==inst_select.value]

obs_options=['all']  
obs_options.extend(data_dates_range3.utc)
obs_select = widget.SelectMultiple(
    options=obs_options,
    value=['all'],
    description='obs selection:',
    style = {'description_width': 'initial'}
)
display(obs_select)

In [ ]:
data_dates_range4=data_dates_range3
if obs_select.value[0]!='all':
    data_dates_range4=data_dates_range3[data_dates_range3.utc.isin(obs_select.value)]

    
all_inst=True if len(unique(data_dates_range4.inst)) >1 else False

data_dates_range,instrument,w0=get_instrument_date_details(data_dates_range4.file,inst_select.value,
                                                           all_inst=all_inst,qgrid=False)#,start_date='2008',end_date='2012')
data_dates_range

In [ ]:
df_av=get_av_spec(data_dates_range,w0,norm=False,output=True,plot_av=False,savefig=False)#,label='utc_inst')
df_av_norm=get_av_spec(data_dates_range,w0,norm=True,output=True,plot_av=False)#,label='utc_inst')
USH.target=data_dates_range.target.iloc[0]
USH.instrument=data_dates_range.inst.iloc[0]

### Template spectral type selection

In [ ]:
spts_templ=['all']  
spts_templ_list=unique(data_dates_range_templ.sp_t).tolist()
spts_templ.extend(spts_templ_list)
templ_select = widget.SelectMultiple(options=spts_templ,
                                     value=['all'],
                                     description='Select SpT(s):',style = {'description_width': 'initial'})
#print('spectral type of ',USH.target,' is:', mk)

print('Sp_t of ',USH.target)
print(unique(data_dates_range4.sp_t))
print(' ')
print('Select spectral types to load in for interactive removals.')
print('For automatic removals select "all" ')
display(templ_select)

In [ ]:
data_dates_range_templ_sel1=data_dates_range_templ
if templ_select.value[0] !='all':
    data_dates_range_templ_sel1=data_dates_range_templ[data_dates_range_templ.sp_t.isin(templ_select.value)]
inst_options=['any']
inst_options.extend(unique(data_dates_range_templ_sel1.inst).tolist())
inst_select = widget.RadioButtons(
    options=inst_options,
    description='Instrument selection of templates:',
    style = {'description_width': 'initial'}
)
display(inst_select)

### Display loaded in templates

In [ ]:
all_inst=True if inst_select.value=='any' else False

#data_dates_range_templ_sel,instrument,w0=get_instrument_date_details(data_dates_range_templ_sel1.file,inst_select.value,
#                                                           all_inst=all_inst,qgrid=False,w_range_auto=True)#,start_date='2008',end_date='2012')
if inst_select.value=='any':
    data_dates_range_templ_sel=data_dates_range_templ_sel1.copy()
else:
    data_dates_range_templ_sel=data_dates_range_templ_sel1[data_dates_range_templ_sel1.inst==inst_select.value]

data_dates_range_templ_sel.sort_values('sp_t',inplace=True)
display(data_dates_range_templ_sel)

print('now loading in templates to dataframe for removals....')

w_min=data_dates_range_templ_sel.wmin.values[0]*10
w_max=data_dates_range_templ_sel.wmax.values[0]*10
w_step=0.01
w0_templ=np.arange(w_min,w_max,w_step)
templ_av=get_av_spec(data_dates_range_templ_sel,w0_templ,norm=False,output=False,plot_av=False,savefig=False,label='spt')
templ_av_norm=get_av_spec(data_dates_range_templ_sel,w0_templ,norm=True,output=False,plot_av=False,label='spt')
offset_wl_plot(templ_av_norm)
xlim(5500,6400),ylim(-3,20)

# Interactive removals

### Select the line and wavelength range to remove photosphere

In [ ]:
line_sel=widget.FloatText(value='6300',step=10,description='Line centre',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
range_sel=widget.FloatText(value='30',step=5,description='Wavelength range (AA)',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

display(widget.HBox([line_sel,range_sel]))

In [ ]:
line=line_sel.value
width_pm=range_sel.value

df_line=get_line_spec(df_av,line,width_pm,norm=False,full_norm=False,cont_sub=False) 
wl_plot(df_line,plot_av=False,fs=USH.fig_size_n,legend=True)
date_selector = widget.Dropdown(options=df_line.columns[1:-1],value=df_line.columns[1],description='Select dates target:',style=dict(description_width='initial'), layout={'width': 'max-content'})

df_line_templ=get_line_spec(templ_av,line,width_pm,norm=True,full_norm=False,cont_sub=False) 

offset_wl_plot(df_line_templ)
date_selector2 = widget.Dropdown(options=df_line_templ.columns[1:-3],value=df_line_templ.columns[1],description='Select dates template:',style=dict(description_width='initial'), layout={'width': 'max-content'})

display(date_selector)
display(date_selector2)

### Initial estimate of vsini and determine RV wrt template

In [ ]:
rv_cen_sel=widget.FloatText(value=line,step=10,description='RV wave centre',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
rv_range_sel=widget.FloatText(value='100',step=5,description='RV Wavelength range (AA)',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

display(widget.HBox([rv_cen_sel,rv_range_sel]))


In [ ]:
templ_name=date_selector2.value.split('_-_')[1]
templ_inst=date_selector2.value.split('_-_')[-1]
target_inst=USH.instrument
data_dates_range_templ_sel2=data_dates_range_templ_sel[data_dates_range_templ_sel.target==templ_name]
st_info,st_wave,st_flux,st_err=read_fits_files(data_dates_range_templ_sel2.file.iloc[0],verbose=True)

if USH.inst_res[templ_inst] > USH.inst_res[target_inst]:
    e_res=effective_res(USH.inst_res[templ_inst],USH.inst_res[target_inst])
else:
    e_res=None

#e_res=None

templ_rv=data_dates_range_templ_sel2.RV.values[0]

radvel_t,vsini=get_rv_vsini(df_av,st_wave,st_flux,st_rv=templ_rv,date=date_selector.value,adj_templ_res=e_res,
                                                    w_min=rv_cen_sel.value-rv_range_sel.value,w_max=rv_cen_sel.value+rv_range_sel.value,
                                                    vsini_max=40,output=True)
#note must set rv of template, st_rv, to get actual rv, otherwise willonly return offset...


# Interactive photosphere removal for above data and template

In [ ]:
#DEFAULT PARAMS
obs_target=date_selector.value
obs_templ=date_selector2.value
def view_subtraction(rv_shift=radvel_t,shift=0,vsini=vsini,r=0,plot_x=[],plot_subtracted=True,
                     auto_r=False,auto_vsini=False,savefig=False,savefits=False,chi_output=True):
    global df_av_sub,params
    #return_params=True requires two values to unpack and also returns the original and template spectra in df_av_sub
    df_av_sub,params=subtract_templ(df_line,obs_target,target_inst,df_line_templ,obs_templ,templ_inst,rv_templ=templ_rv,
                                    rv_shift=rv_shift,vsini=vsini,r=r,fs=USH.fig_size_l,plot_x=plot_x,mask_pm=[None,2],
                                 shift=shift,plot_subtracted=plot_subtracted,plot_divided=False,
                                    return_params=True,auto_r=auto_r,auto_vsini=auto_vsini,
                                   output=True,savefig=savefig,savefits=savefits,chi_output=chi_output)
interact(view_subtraction, 
         rv_shift=widget.FloatSlider(min=-100, max=100, step=0.01,value=radvel_t,layout={'width': '800px'},continuous_update=False),
         shift=widget.FloatSlider(min=-0.2, max=0.2, step=0.001,value=0,layout={'width': '800px'},continuous_update=False), 
         vsini=widget.FloatSlider(min=0.01, max=50, step=0.01,value=vsini,layout={'width': '800px'},continuous_update=False), 
         #factor=widget.FloatSlider(min=0.01, max=5, step=0.01,value=1,layout={'width': '800px'},continuous_update=False), 
         r=widget.FloatSlider(min=-0.99, max=10, step=0.1,value=0,layout={'width': '800px'},continuous_update=False),
         plot_x=widget.FloatRangeSlider(min=min(df_line.wave),max=max(df_line.wave),step=1,value=[min(df_line.wave),max(df_line.wave)],layout={'width': '800px'},continuous_update=False)
        )



In [ ]:
#df_av_sub=pd.DataFrame({'wave': w0_subtracted,'before':df_line[obs],'after':f0_subtracted,'av_flux':f0_subtracted,'med_flux':f0_subtracted,'std_flux':f0_subtracted})
#df_av_sub=pd.DataFrame({'wave': w0_subtracted,'before':df_line[obs],'after':f0_subtracted,'av_flux':f0_subtracted,'med_flux':f0_subtracted,'std_flux':f0_subtracted})

wl_plot(df_av_sub,plot_av=False,legend=True,fs=USH.fig_size_n)
df_av_sub_vel=get_line_spec(df_av_sub,line,w_range=450,vel_offset=0,vel=True)
vel_plot(df_av_sub_vel,line=line,plot_av=False,legend=True)
min(df_av_sub.wave),len(df_av_sub)

# Automatic removals for loaded in target and selected templates above

### The following data will be used

In [ ]:
obs_target=date_selector.value
print('selected spectra: ',obs_target)
wl_plot(df_line,plot_av=False,fs=USH.fig_size_n,legend=True)
offset_wl_plot(df_line_templ)


In [ ]:
%%time

#this goes through only the df_line for the selected target loaded in, not all data...
# but this has the implementation to only loop through templates within a sp_t tolerance set to 2.0 classes.

#requires df_av to be dataframe of target observations
# df_line_templ is selection of templates for given target 
target_inst=USH.instrument
df_line=get_line_spec(df_av,line,width_pm,norm=True,full_norm=False,cont_sub=False)
obs_target=date_selector.value

spt_code_target=spt_coding(data_dates_range4.sp_t.values[0]) 

df_line_templ=get_line_spec(templ_av,line,width_pm,norm=True,full_norm=False,cont_sub=False) 
obs_templ_list=df_line_templ.columns[1:-3]

sub_params_loop=pd.DataFrame()

for obs_templ in obs_templ_list:#[0:1]:

    templ_name=obs_templ.split('_-_')[1]
    templ_inst=obs_templ.split('_-_')[-1]
    templ_spt=obs_templ.split('_-_')[0]
    spt_code_template=spt_coding(templ_spt)

    if abs(spt_code_target - spt_code_template) <= 2.0:
        
        #read in template fits file for RV/vsini calculation
        data_dates_range_templ_sel2=data_dates_range_templ_sel[data_dates_range_templ_sel.target==templ_name]
        st_info,st_wave,st_flux,st_err=read_fits_files(data_dates_range_templ_sel2.file.iloc[0],verbose=True)
        #calculate rv and vsini wrt to template
        
        if USH.inst_res[templ_inst] > USH.inst_res[target_inst]:
            e_res=effective_res(USH.inst_res[templ_inst],USH.inst_res[target_inst])
        else:
            e_res=None
        
        templ_rv=data_dates_range_templ_sel2.RV.values[0]
        
        radvel_t,vsini_t=get_rv_vsini(df_av,st_wave,st_flux,st_rv=templ_rv,date=date_selector.value,adj_templ_res=e_res,
                                                            w_min=line-100,w_max=line+100,vsini_max=80,output=False)
        #run phot.sub. with best fit values
        df_av_sub,params=subtract_templ(df_line,obs_target,target_inst,df_line_templ,obs_templ,templ_inst,rv_templ=templ_rv,
                                            rv_shift=radvel_t,vsini=vsini_t,r=0,fs=USH.fig_size_l,plot_x=[],mask_pm=[None,2],
                                         shift=0,plot_subtracted=True,plot_divided=False,
                                            return_params=True,auto_r=True,auto_vsini=True,chi_output=True,
                                           output=True,savefig=False,savefits=False)
        sub_params_loop = pd.concat([sub_params_loop, pd.DataFrame([params])], ignore_index=True)


dirname='sub_params'
filename='phot_sub_params_OI_Vary.csv'
timenow_bu=time.strftime("%d_%b_%Y_%H_%M", time.gmtime())
if os.path.exists(os.path.join(dirname,filename)):
    os.system('cp %s backups/%s'%(os.path.join(dirname,filename),timenow_bu+filename))
#sub_params_loop.to_csv(os.path.join(dirname,filename), mode='a', header=not os.path.exists(os.path.join(dirname,filename)),index=False)
print('saving results to output file: ',filename)

In [ ]:
close('all') #if too many plots are open

# Automatic removals for all data in data_dir and all templates given sp_t tolerance

In [ ]:
print('loading all templates...')
w_min=3770#data_dates_range_templ_sel.wmin.values[0]*10
w_max=7900#data_dates_range_templ_sel.wmax.values[0]*10
w_step=0.01
w0_templ=np.arange(w_min,w_max,w_step)
data_dates_range_templ_cl3=data_dates_range_templ#[data_dates_range_templ.templ=='Cl3']
templ_av=get_av_spec(data_dates_range_templ_cl3,w0_templ,norm=False,output=False,plot_av=False,savefig=False,label='spt')
obs_templ_list=templ_av.columns[1:-3]
print('done')

In [ ]:
obs_templ_list.sort_values()

In [ ]:
data_dates_range2_ori=data_dates_range2

In [ ]:
print('all loaded in target data:...')
#data_dates_range2=data_dates_range2_ori.iloc[10:14]
data_dates_range2

In [ ]:
inst_options=['any']
inst_options.extend(unique(data_dates_range2.inst).tolist())
inst_select_tar = widget.SelectMultiple(options=inst_options,value=['any'],description='Instrument selection of target:',style = {'description_width': 'initial'})

line_sel=widget.FloatText(value='6300',step=10,description='Line centre',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
range_sel=widget.FloatText(value='30',step=5,description='Wavelength range (AA)',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

rv_cen_sel=widget.FloatText(value=6300,step=10,description='RV wave centre',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
rv_range_sel=widget.FloatText(value='100',step=5,description='RV Wavelength range (AA)',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

spt_sel=widget.FloatSlider(min=0.0, max=5, step=0.5,value=2,description='SpT range +/-:', style = {'description_width': 'initial'})


savefig=widget.Checkbox(value=True,description='Save all results plots?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
savefits=widget.Checkbox(value=True,description='Save all results .fits files?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
filename_sel=widget.Text(value='loop_tests_rv.csv',description='enter filename for output info:',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
displayplots=widget.Checkbox(value=False,description='Show all plots in notebook?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
displaychi2=widget.Checkbox(value=False,description='Show chi sq. plots in notebook?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

display(inst_select_tar)
display(spt_sel)
display(widget.HBox([line_sel,range_sel]))
display(widget.HBox([rv_cen_sel,rv_range_sel]))
display(widget.HBox([filename_sel,savefig,savefits]))
display(widget.HBox([displayplots,displaychi2]))

In [ ]:
%%time
#currently, this loops through all data initally loaded into the notebook at the start
#for all templates loaded into df_line_templ from the spectral type selection at that part
#can set the sp_t tolerance for looping through
#you can then search through the sub_params data_frame/csv file to find the lowest chisq or eud_d per target...
#this is the full auto removal loop, which will run through all data in the data_dates_range1 dataframe

print(f"About to run full auto removal for line: '{line_sel.value}' with template RV calculation at: '{rv_cen_sel.value}'")
print(f"Current output filename is '{filename_sel.value}'. ")
confirmation = input(f"Is this correct? (y/yes): ").lower()
if confirmation not in ['y', 'yes']:
    raise ValueError("Notebook execution stopped. Please confirm the filename.")

line=line_sel.value
width_pm=range_sel.value #in AA


if 'any' in inst_select_tar.value:
    data_dates_range1 = data_dates_range2
else:
    data_dates_range1 = data_dates_range2[data_dates_range2['inst'].isin(inst_select_tar.value)]

#remove the non TELL CORR UVES data...
data_dates_range1 = data_dates_range1[~((data_dates_range1['inst'] == 'UVES') & (data_dates_range1['file'].str.contains('REDU.fits')))]
data_dates_range1 = data_dates_range1[~((data_dates_range1['inst'] == 'UVES') & (data_dates_range1['file'].str.contains('redu.fits')))]

sub_params=pd.DataFrame()

#for index, row in data_dates_range1.head(15).iterrows(): #for checking the loop before running it on everything, can change to e.g. head(2) for first two starrs in list
for index, row in data_dates_range1.iterrows():
    try:
        data_dates_range5=pd.DataFrame([row])
        USH.target=data_dates_range5.target.values[0]
        USH.instrument=data_dates_range5.inst.values[0]
        target_inst=USH.instrument
        w_min=data_dates_range5.wmin.values[0]*10
        w_max=data_dates_range5.wmax.values[0]*10
        w_step=0.01
        if (w_min < line) & (w_max > line):
            w0=np.arange(w_min,w_max,w_step)
            df_av=get_av_spec(data_dates_range5,w0,norm=False,output=False,plot_av=False,savefig=False)#,label='utc_inst')
        
            df_line=get_line_spec(df_av,line,width_pm,norm=True,full_norm=False,cont_sub=False)
            obs_target=df_line.columns[1]
    
            spt_code_target=spt_coding(data_dates_range5.sp_t.values[0])
            
        
            #select template (to be looped)
            for obs_templ in obs_templ_list:#[0:2]:
    
                templ_spt=obs_templ.split('_-_')[0]
                templ_name=obs_templ.split('_-_')[1]
                templ_inst=obs_templ.split('_-_')[-1]
                spt_code_template=spt_coding(templ_spt)
    
                if abs(spt_code_target - spt_code_template) <= spt_sel.value:
                    #read in template fits file for RV/vsini calculation
                    data_dates_range_templ_sel2=data_dates_range_templ[data_dates_range_templ.target==templ_name]
                    st_info,st_wave,st_flux,st_err=read_fits_files(data_dates_range_templ_sel2.file.iloc[0],verbose=True)
                    #calculate rv and vsini wrt to template
                    rv_wl_min=rv_cen_sel.value-rv_range_sel.value
                    rv_wl_max=rv_cen_sel.value+rv_range_sel.value

                    if USH.inst_res[templ_inst] > USH.inst_res[target_inst]:
                        e_res=effective_res(USH.inst_res[templ_inst],USH.inst_res[target_inst])
                    else:
                        e_res=None
                    
                    templ_rv=data_dates_range_templ_sel2.RV.values[0]
                    
                    radvel_t,vsini_t=get_rv_vsini(df_av,st_wave,st_flux,st_rv=templ_rv,date=obs_target,adj_templ_res=e_res,
                                                                        w_min=rv_wl_min,w_max=rv_wl_max,vsini_max=80,output=False)
                    df_line_templ=get_line_spec(templ_av,line,width_pm,norm=True,full_norm=False,cont_sub=False) 
                    #run phot.sub. with best fit values
                    df_av_sub,params=subtract_templ(df_line,obs_target,target_inst,df_line_templ,obs_templ,templ_inst,rv_templ=templ_rv,
                                                    rv_shift=radvel_t,vsini=vsini_t,r=0,fs=USH.fig_size_l,plot_x=[],mask_pm=[None,2],
                                                 shift=0,plot_subtracted=True,plot_divided=False,
                                                    return_params=True,auto_r=True,auto_vsini=True,chi_output=displaychi2.value,
                                                   output=displayplots.value,savefig=savefig.value,savefits=savefits.value)
                    sub_params = pd.concat([sub_params, pd.DataFrame([params])], ignore_index=True)
    except:
        continue

dirname='sub_params'
filename=filename_sel.value #'loop_tests.csv'###########################
timenow_bu=time.strftime("%d_%b_%Y_%H_%M", time.gmtime())
if os.path.exists(os.path.join(dirname,filename)):
    os.system('cp %s backups/%s'%(os.path.join(dirname,filename),timenow_bu+filename))
sub_params.to_csv(os.path.join(dirname,filename), mode='a', header=not os.path.exists(os.path.join(dirname,filename)),index=False)
print('saving results to output file: ',filename)
print(len(sub_params),' removals performed')
    


In [ ]:
for target in unique(data_dates_range1.target):
    print(target)

In [ ]:
#cbeck list of targets names vs output file


In [ ]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')

In [ ]:
close('all')

### Simple fitting


In [ ]:
radvel=0

In [ ]:
line_select = widget.FloatText(value=line,step=1,description='Line wavelength (A)',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
rv_select = widget.FloatText(value=0,step=1,description='Star Radial Velocity (km/s)',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

date_selector = widget.SelectMultiple(options=df_av.columns[1:-1],value=['med_flux'],description='Select dates:',)
display(widget.HBox([line_select,rv_select,date_selector]))

vel_range=widget.FloatText(value='350',step=10,description='Velocity window',layout={'width': 'max-content'}, style = {'description_width': 'initial'})


ngauss_sel=widget.IntSlider(value='1',min=1,max=3,description='# of positive Gauss:', style = {'description_width': 'initial'})
neg_sel=widget.Checkbox(value=False,description='Include a negative Gauss?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
vred_sel=widget.Checkbox(value=False,description='Include a Vred calc.?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
gof_min_sel=widget.FloatSlider(min=0.1, max=1, step=0.1,value=0.2,description='GoF min:', style = {'description_width': 'initial'})
reject_low_gof=widget.Checkbox(value=False,description='Use min GoF?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
reject_line_close=widget.Checkbox(value=False,description='Use min std err?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

set_g1_cen=widget.Checkbox(value=False,description='Set G1 centre limits?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
set_g1_sig=widget.Checkbox(value=False,description='Set G1 sigma limits?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g1_cen_min=widget.FloatText(value='0',step=10,description='G1 centre min',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g1_cen_max=widget.FloatText(value='20',step=10,description='max',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g1_sig_min=widget.FloatText(value='0',step=10,description='G1 sigma min',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g1_sig_max=widget.FloatText(value='50',step=10,description='max',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

set_g2_cen=widget.Checkbox(value=False,description='Set G2 centre limits?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
set_g2_sig=widget.Checkbox(value=False,description='Set G2 sigma limits?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g2_cen_min=widget.FloatText(value='10',step=10,description='G2 centre min',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g2_cen_max=widget.FloatText(value='50',step=10,description='max',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g2_sig_min=widget.FloatText(value='0',step=10,description='G2 sigma min',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g2_sig_max=widget.FloatText(value='100',step=10,description='max',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

set_g3_cen=widget.Checkbox(value=False,description='Set G3 centre limits?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
set_g3_sig=widget.Checkbox(value=False,description='Set G3 sigma limits?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g3_cen_min=widget.FloatText(value='10',step=10,description='G3 centre min',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g3_cen_max=widget.FloatText(value='50',step=10,description='max',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g3_sig_min=widget.FloatText(value='0',step=10,description='G3 sigma min',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g3_sig_max=widget.FloatText(value='100',step=10,description='max',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

set_g4_cen=widget.Checkbox(value=False,description='Set neg Gauss centre limits?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
set_g4_sig=widget.Checkbox(value=False,description='Set neg Gauss sigma limits?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g4_cen_min=widget.FloatText(value='10',step=10,description='neg centre min',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g4_cen_max=widget.FloatText(value='50',step=10,description='max',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g4_sig_min=widget.FloatText(value='0',step=10,description='neg sigma min',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
g4_sig_max=widget.FloatText(value='100',step=10,description='max',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

savefig=widget.Checkbox(value=False,description='Save all output plots to file?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
title=widget.Dropdown(options=['full','simple','none'],value='full',description='Plot title',layout={'width': 'max-content'}, style = {'description_width': 'initial'})
df_sel=widget.Dropdown(options=['df_av','df_av_sub','df_av_norm'],value='df_av_sub',description='Use df_av or df_av_norm?',layout={'width': 'max-content'}, style = {'description_width': 'initial'})

display(df_sel,vel_range)
display(widget.TwoByTwoLayout(top_left=widget.HBox([reject_low_gof,reject_line_close]),top_right=gof_min_sel,bottom_left=ngauss_sel,bottom_right=widget.HBox([neg_sel,vred_sel])))
display(widget.TwoByTwoLayout(bottom_left=widget.HBox([g1_cen_min,g1_cen_max]),bottom_right=widget.HBox([g1_sig_min,g1_sig_max]),top_left=set_g1_cen,top_right=set_g1_sig))
display(widget.TwoByTwoLayout(bottom_left=widget.HBox([g2_cen_min,g2_cen_max]),bottom_right=widget.HBox([g2_sig_min,g2_sig_max]),top_left=set_g2_cen,top_right=set_g2_sig))
display(widget.TwoByTwoLayout(bottom_left=widget.HBox([g3_cen_min,g3_cen_max]),bottom_right=widget.HBox([g3_sig_min,g3_sig_max]),top_left=set_g3_cen,top_right=set_g3_sig))
display(widget.TwoByTwoLayout(bottom_left=widget.HBox([g4_cen_min,g4_cen_max]),bottom_right=widget.HBox([g4_sig_min,g4_sig_max]),top_left=set_g4_cen,top_right=set_g4_sig))
display(widget.HBox([title,savefig]))
    

* *The next block runs the fitting for the lines and options selected above*
* *The option to output each plot on screen is given*


## Fit the lines

In [ ]:
line_date_list=date_selector.value
line=line_select.value
radvel=rv_select.value
print('attempting fit of line at ',line)
print('number of dates attempting to fit %s'%(len(line_date_list)))

#settings for Vred H balmer: g1 cen -75,50, neg cen 150,250, neg sig 0,250
#settings for Vred Ca II k: g1 cen -20,20, g2 cen -30,20 neg cen 150,250, neg sig 0,250

g1_cen=[g1_cen_min.value,g1_cen_max.value] if set_g1_cen.value==True else None #0,150 for CaK + H
g2_cen=[g2_cen_min.value,g2_cen_max.value] if set_g2_cen.value==True else None #-20,100 for CaK
g3_cen=[g3_cen_min.value,g3_cen_max.value] if set_g3_cen.value==True else None #-20,100 for CaK
neg_cen=[g4_cen_min.value,g4_cen_max.value] if set_g4_cen.value==True else None #0,50 for He, 200,50 for CaK + H
g1_sig=[g1_sig_min.value,g1_sig_max.value] if set_g1_sig.value==True else None #0,50 for He, 200,50 for CaK + H
g2_sig=[g2_sig_min.value,g2_sig_max.value] if set_g2_sig.value==True else None #0,50 for He, 200,50 for CaK + H
g3_sig=[g3_sig_min.value,g3_sig_max.value] if set_g3_sig.value==True else None #0,50 for He, 200,50 for CaK + H
neg_sig=[g4_sig_min.value,g4_sig_max.value] if set_g4_sig.value==True else None #0,50 for He, 200,50 for CaK + H

df_fit=df_av if df_sel.value=='df_av' else df_av_norm if df_sel.value=='df_av_norm' else df_av_sub

df_av_line_vel=get_line_spec(df_fit,line,vel_range.value,vel_offset=radvel,vel=True)
line_results=pd.DataFrame()

# df_av_line_vel=df_av_line_vel.query('vel < 43 or vel > 74')
# df_av_line_vel=df_av_line_vel.query('vel < 95 or vel > 130')
# df_av_line_vel=df_av_line_vel.query('vel < -203 or vel > -140')
# #df_av_line_vel=df_av_line_vel.query('vel < 135 or vel > 160')
# #df_av_line_vel=df_av_line_vel.query('vel < 226 or vel > 291')


#df_av_line_vel=df_av_line_vel.rolling(5).mean().dropna()


output=True#show_output('individual em lines plots?')

for date in date_selector.value:
    # try:
    #     target=data_dates_range[data_dates_range.utc==date.split('_')[0]].target.values[0]
    # except:
    #     target=data_dates_range[data_dates_range.mjd==float(date)].target.values[0]
    out,x,y,line_info=gauss_stats(df_av_line_vel,date,em_row=line,target=target,
                                  ngauss=ngauss_sel.value,output=output,
                                g1_cen=g1_cen,g2_cen=g2_cen,g3_cen=g3_cen,neg_cen=neg_cen,g1_sig=g1_sig,g2_sig=g2_sig,g3_sig=g3_sig,neg_sig=neg_sig,
                                neg=neg_sel.value,gof_min=gof_min_sel.value,vred=vred_sel.value,
                                  plot_comps=True,title=title.value,legend=True,sub_cont_fit=True,
                                savefig=savefig.value,reject_low_gof=reject_low_gof.value,reject_line_close=reject_line_close.value)
    line_results=pd.concat([line_results,line_info],axis=1,ignore_index=True)
line_results=line_results.T    

#int_flux_results=line_results[['target', 'mjd', 'gof', 'int_flux','EW','asym']]
#int_flux_results['ngauss']=1

#filename='PEN_Ha_fits.csv'
#int_flux_results.to_csv(filename, mode='a', header=not os.path.exists(filename),index=False)



In [ ]:
line_results

In [ ]:
out